# 04 · Prompt Optimization

Compare prompt styles using the same 15 questions from Notebook 3. Test structured answers, question rewriting, and difficult questions.

In [1]:
%pip install -q "chromadb>=1,<2" sentence-transformers openai python-dotenv pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Setup

Use the same embedding provider as Notebook 2. An OpenAI API key is required for generating answers.

In [2]:
import json
import os
import time
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

ROOT = Path.cwd()
BASE = ROOT.parent if ROOT.name == 'notebooks' else ROOT
load_dotenv(BASE / '.env', encoding='utf-8-sig')
load_dotenv(BASE / 'notebooks' / '.env', encoding='utf-8-sig')

MODELS = BASE / 'models'
REPORTS = BASE / 'reports' / 'prompt_optimization'
REPORTS.mkdir(parents=True, exist_ok=True)

EMBEDDING_PROVIDER = 'local'  # 'local' or 'openai'
CHAT_MODEL = 'gpt-4o-mini'     # change if you prefer another chat model
TOP_K = 5
LOCAL_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'
LOCAL_WINDOW_TOKENS = 220   # must match Notebook 2

if not os.getenv('OPENAI_API_KEY'):
    raise ValueError('Add OPENAI_API_KEY to the project .env file before running Step 3.')
if EMBEDDING_PROVIDER not in ('local', 'openai'):
    raise ValueError('EMBEDDING_PROVIDER must be local or openai.')

llm = OpenAI(max_retries=3, timeout=60)
store = chromadb.PersistentClient(path=str(MODELS / 'chroma'))
collection_name = f'bank_filings_{EMBEDDING_PROVIDER}'
try:
    collection = store.get_collection(collection_name, embedding_function=None)
except Exception as exc:
    raise ValueError(f'{collection_name} not found. Run Notebook 2 with this embedding provider.') from exc

if collection.count() == 0:
    raise ValueError('The selected ChromaDB collection is empty. Run Notebook 2 first.')
print(f'Using {collection_name}: {collection.count():,} chunks')
print(f'Answer model: {CHAT_MODEL}')

Using bank_filings_local: 10,196 chunks
Answer model: gpt-4o-mini


## 2. Retrieval

Reuse the same embedding method as Notebook 3. This keeps the comparison fair.

In [3]:
def normalize(vectors):
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.maximum(lengths, 1e-12)


if EMBEDDING_PROVIDER == 'local':
    local_model = SentenceTransformer(LOCAL_MODEL)


def embed_questions(questions):
    if EMBEDDING_PROVIDER == 'openai':
        response = llm.embeddings.create(model=OPENAI_EMBEDDING_MODEL, input=questions)
        ordered = sorted(response.data, key=lambda item: item.index)
        return normalize(np.array([item.embedding for item in ordered], dtype=np.float32))

    tokenizer = local_model.tokenizer
    window_size = min(LOCAL_WINDOW_TOKENS, local_model.max_seq_length - 2)
    windows, owners, weights = [], [], []
    for index, question in enumerate(questions):
        tokens = tokenizer.encode(question, add_special_tokens=False)
        for start in range(0, len(tokens), window_size):
            piece = tokens[start:start + window_size]
            windows.append(tokenizer.decode(piece))
            owners.append(index)
            weights.append(len(piece))
    if not windows:
        raise ValueError('Questions cannot be empty.')
    window_vectors = local_model.encode(windows, batch_size=32, normalize_embeddings=True)
    vectors = np.zeros((len(questions), window_vectors.shape[1]), dtype=np.float32)
    np.add.at(vectors, owners, window_vectors * np.array(weights)[:, None])
    return normalize(vectors)


def make_filter(**fields):
    conditions = [{key: {'$eq': value}} for key, value in fields.items() if value is not None]
    if not conditions:
        return None
    return conditions[0] if len(conditions) == 1 else {'$and': conditions}


def retrieve(question, filters=None, k=TOP_K):
    vector = embed_questions([question])[0]
    results = collection.query(
        query_embeddings=[vector.tolist()],
        n_results=min(k, collection.count()),
        where=make_filter(**(filters or {})),
        include=['documents', 'metadatas', 'distances'],
    )
    return [
        {
            'chunk_id': chunk_id,
            'text': text,
            'similarity': round(1 - distance, 4),
            **metadata,
        }
        for chunk_id, text, distance, metadata in zip(
            results['ids'][0], results['documents'][0],
            results['distances'][0], results['metadatas'][0]
        )
    ]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Questions and source labels

All prompt styles will receive the same retrieved passages for each question.

In [4]:
questions = [
    ('RBC', 'credit', 'What credit risks and loan-loss provisions does RBC discuss?'),
    ('RBC', 'market', 'How does RBC describe its market risk exposure?'),
    ('RBC', 'climate', 'What climate-related risks does RBC identify?'),
    ('TD', 'operational', 'What cybersecurity and operational risks does TD discuss?'),
    ('TD', 'credit', 'What risks does TD identify in its residential mortgage lending?'),
    ('TD', 'liquidity', 'How does TD discuss liquidity and funding risk?'),
    ('BMO', 'liquidity', 'What liquidity risk management measures does BMO describe?'),
    ('BMO', 'market', 'What does BMO say about interest-rate risk?'),
    ('BMO', 'operational', 'What operational risks does BMO disclose?'),
    ('BNS', 'climate', 'What physical and transition climate risks does Scotiabank discuss?'),
    ('BNS', 'credit', 'What credit risks does Scotiabank report?'),
    ('BNS', 'market', 'What market risks does Scotiabank describe?'),
    ('CIBC', 'credit', 'What housing and mortgage credit risks does CIBC identify?'),
    ('CIBC', 'operational', 'What cyber and technology-related risks does CIBC disclose?'),
    ('CIBC', 'liquidity', 'How does CIBC describe its liquidity risk management?'),
]

BANK_NAMES = {'RBC': 'RBC', 'TD': 'TD', 'BMO': 'BMO', 'BNS': 'Scotiabank', 'CIBC': 'CIBC'}
tests = pd.DataFrame([
    {'id': f'q{i:02d}', 'bank_code': bank, 'bank': BANK_NAMES[bank],
     'risk': risk, 'question': question}
    for i, (bank, risk, question) in enumerate(questions, start=1)
])
display(tests)

,id,bank_code,bank,risk,question
0,q01,RBC,RBC,credit,What credit risks and loan-loss provisions doe...
1,q02,RBC,RBC,market,How does RBC describe its market risk exposure?
2,q03,RBC,RBC,climate,What climate-related risks does RBC identify?
3,q04,TD,TD,operational,What cybersecurity and operational risks does ...
4,q05,TD,TD,credit,What risks does TD identify in its residential...
5,q06,TD,TD,liquidity,How does TD discuss liquidity and funding risk?
6,q07,BMO,BMO,liquidity,What liquidity risk management measures does B...
7,q08,BMO,BMO,market,What does BMO say about interest-rate risk?
8,q09,BMO,BMO,operational,What operational risks does BMO disclose?
9,q10,BNS,Scotiabank,climate,What physical and transition climate risks doe...


In [5]:
def source_name(hit):
    quarter = hit.get('quarter')
    period = f' Q{quarter}' if quarter not in (None, '', 0, '0') else ''
    return (f"{hit.get('bank', hit.get('bank_code', 'Unknown bank'))} | "
            f"{hit.get('document_type', 'filing')} | "
            f"{hit.get('fiscal_year', 'unknown year')}{period} | "
            f"{hit.get('section') or 'Unspecified section'} | "
            f"pages {hit.get('page_start', '?')}-{hit.get('page_end', '?')}")


def build_context(hits):
    return '\n\n'.join(
        f"[S{i}] {source_name(hit)}\nChunk ID: {hit['chunk_id']}\n{hit['text']}"
        for i, hit in enumerate(hits, 1)
    )


passages = {}
for test in tqdm(tests.to_dict('records'), desc='Retrieving passages'):
    passages[test['id']] = retrieve(test['question'], {'bank_code': test['bank_code']})

print('Retrieved passages for', len(passages), 'questions')

Retrieving passages:   0%|          | 0/15 [00:00<?, ?it/s]

Retrieved passages for 15 questions


## 4. Compare three prompt styles

**Basic:** short grounded instructions. **Few-shot:** gives examples of proper answers. **Step-by-step:** asks the model to check relevant evidence before composing a concise final answer. We do not request private reasoning.

In [6]:
COMMON_RULES = """You are a Canadian bank compliance research assistant.
Use only the filing excerpts supplied. Do not invent facts, numbers, or citations.
Cite factual claims using [S1], [S2], etc., matching the supplied labels.
If evidence is missing, say: The provided filings do not contain enough information to answer this question.
Never combine data from different years without explaining the difference."""

PROMPTS = {
    'basic': COMMON_RULES,
    'few_shot': COMMON_RULES + """
Example question: What risks does the filing describe?
Example excerpt: [S1] Example Bank 2025 Annual Report: The bank faces cyber and credit risks.
Example answer: The filing identifies cybersecurity and credit risk [S1].

Example question: What was the exact amount of provisions?
Example excerpt: [S1] Example Bank 2025 Annual Report: The bank describes credit risk, but gives no provision amount.
Example answer: The provided filings do not contain enough information to answer this question.
""",
    'step_by_step': COMMON_RULES + """
Before answering, check that each claim is supported by a specific excerpt.
Check bank names, time periods, and any reported figures.
Write only the final answer with citations, not your internal reasoning.
Mention missing evidence clearly.
""",
}


def ask_model(question, hits, style='basic', json_output=False):
    if not hits:
        return {'answer': 'No matching passages were retrieved.', 'sources': [], 'input_tokens': 0, 'output_tokens': 0}
    messages = [
        {'role': 'system', 'content': PROMPTS[style]},
        {'role': 'user', 'content': f'FILING EXCERPTS:\n{build_context(hits)}\n\nQUESTION:\n{question}'},
    ]
    args = {'model': CHAT_MODEL, 'temperature': 0, 'messages': messages, 'max_completion_tokens': 1200}
    if json_output:
        messages[0]['content'] += """
Return a JSON object with fields: answer (string), citations (list of source labels),
sources (list of objects with label, bank, document, section, year copied exactly from each cited excerpt),
confidence (low, medium, or high), caveats (list of strings).
Citations must use only labels found in the excerpts. Return no other text."""
        args['response_format'] = {'type': 'json_object'}
    started = time.time()
    response = llm.chat.completions.create(**args)
    return {
        'answer': response.choices[0].message.content or '',
        'sources': [{'label': f'S{i}', **hit} for i, hit in enumerate(hits, 1)],
        'seconds': round(time.time() - started, 2),
        'input_tokens': response.usage.prompt_tokens if response.usage else None,
        'output_tokens': response.usage.completion_tokens if response.usage else None,
        'finish_reason': response.choices[0].finish_reason,
    }

## 5. Run the comparison

This makes up to 45 chat requests (15 questions × 3 styles). Saved results are reused on later runs. Delete the output file or set `RERUN = True` to rerun.

In [7]:
RESULTS_PATH = REPORTS / f'prompt_comparison_{EMBEDDING_PROVIDER}.json'
RERUN = False

if RESULTS_PATH.exists() and not RERUN:
    comparisons = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
else:
    comparisons = []
    for test in tqdm(tests.to_dict('records'), desc='Comparing prompts'):
        for style in PROMPTS:
            result = ask_model(test['question'], passages[test['id']], style)
            comparisons.append({**test, 'style': style, **result})
            RESULTS_PATH.write_text(json.dumps(comparisons, indent=2, ensure_ascii=False), encoding='utf-8')

comparison_table = pd.DataFrame([
    {key: row.get(key) for key in ('id', 'bank', 'question', 'style', 'answer', 'seconds', 'input_tokens', 'output_tokens')}
    for row in comparisons
])
comparison_table.to_csv(REPORTS / f'prompt_comparison_{EMBEDDING_PROVIDER}.csv', index=False)
display(comparison_table[['id', 'style', 'answer']].head(9))

,id,style,answer
0,q01,basic,Royal Bank of Canada (RBC) discusses several a...
1,q01,few_shot,The filing discusses several aspects of credit...
2,q01,step_by_step,Royal Bank of Canada (RBC) discusses several a...
3,q02,basic,RBC describes its market risk exposure through...
4,q02,few_shot,The filing describes RBC's market risk exposur...
5,q02,step_by_step,RBC describes its market risk exposure through...
6,q03,basic,RBC identifies climate-related risks as being ...
7,q03,few_shot,The filing identifies two main categories of c...
8,q03,step_by_step,RBC identifies climate-related risks as transi...


## 6. Review answers and citations

Change the question ID to see all three prompt styles alongside the passages they cited. Evaluate whether the filing actually answers the question, whether each answer is correct, and whether its citations support every claim.

In [8]:
QUESTION_ID = 'q01'
for row in comparisons:
    if row['id'] == QUESTION_ID:
        print('\nSTYLE:', row['style'])
        print('QUESTION:', row['question'])
        print('ANSWER:', row['answer'])

print('\nRETRIEVED PASSAGES:')
for i, hit in enumerate(passages.get(QUESTION_ID, []), 1):
    print(f"\n[S{i}] {source_name(hit)}\nChunk ID: {hit['chunk_id']}\n{hit['text'][:1200]}")


STYLE: basic
QUESTION: What credit risks and loan-loss provisions does RBC discuss?
ANSWER: Royal Bank of Canada (RBC) discusses several aspects of credit risks and loan-loss provisions in their filings:

1. **Credit Risk Management**: RBC establishes risk limits at various levels, including single name limits, regional, country, and industrial sector limits, as well as regulatory large exposure limits. They actively manage credit exposures to align with their risk appetite and to avoid undue concentration risk, which is defined as the risk arising from large exposures that are highly correlated and could be similarly affected by changes in economic or political conditions [S1].

2. **Loan Forbearance**: RBC has specialized groups and formalized policies for managing high-risk, delinquent, or defaulted borrowers. They strive to identify borrowers in financial difficulty early and may modify loan terms to minimize losses. Forbearance agreements can include concessions such as rate redu

In [9]:
REVIEW_PATH = REPORTS / f'prompt_review_{EMBEDDING_PROVIDER}.csv'
if not REVIEW_PATH.exists():
    review = comparison_table[['id', 'bank', 'question', 'style']].copy()
    review['answerable'] = ''             # yes / no / uncertain
    review['answer_correct'] = ''         # yes / partial / no
    review['citations_correct'] = ''      # yes / partial / no
    review['notes'] = ''
    review.to_csv(REVIEW_PATH, index=False)
    print('Created:', REVIEW_PATH)
else:
    print('Review file exists; your ratings were preserved:', REVIEW_PATH)

Review file exists; your ratings were preserved: c:\Users\ashym\Documents\GitHub\jarvis_data_eng_YashMehrotra\ml_3\reports\prompt_optimization\prompt_review_local.csv


## 7. Summarize manual ratings

Fill in the review CSV first. Blank and uncertain ratings are not counted as correct. Scores are descriptive; they do not prove the question was answerable.

In [10]:
review = pd.read_csv(REVIEW_PATH, keep_default_na=False)
for column in ['answerable', 'answer_correct', 'citations_correct']:
    print('\n', column)
    display(pd.crosstab(review['style'], review[column]))

rated = review[review['answer_correct'].isin(['yes', 'partial', 'no'])].copy()
if len(rated):
    rated['correct_score'] = rated['answer_correct'].map({'yes': 1, 'partial': 0.5, 'no': 0})
    display(rated.groupby('style')['correct_score'].agg(['mean', 'count']))
else:
    print('No answer correctness ratings yet.')


 answerable


answerable,
style,
basic,15
few_shot,15
step_by_step,15



 answer_correct


answer_correct,
style,
basic,15
few_shot,15
step_by_step,15



 citations_correct


citations_correct,
style,
basic,15
few_shot,15
step_by_step,15


No answer correctness ratings yet.


## 8. Structured JSON answers

JSON makes citations and caveats easier to process. Check that the labels in the JSON match the retrieved sources. A self-reported confidence value is not a measured accuracy score.

In [11]:
example = tests.iloc[0]
json_result = ask_model(example['question'], passages[example['id']], style='step_by_step', json_output=True)
try:
    structured = json.loads(json_result['answer'])
    assert isinstance(structured.get('answer'), str)
    assert isinstance(structured.get('citations'), list)
    assert isinstance(structured.get('sources'), list)
    source_map = {f'S{i}': hit for i, hit in enumerate(passages[example['id']], 1)}
    assert {s['label'] for s in structured['sources']} == set(structured['citations'])
    for source in structured['sources']:
        hit = source_map[source['label']]
        assert source['bank'] == hit['bank']
        assert source['document'] == hit['document_type']
        assert source['section'] == hit['section']
        assert source['year'] == hit['fiscal_year']
    assert isinstance(structured.get('caveats'), list)
    assert structured.get('confidence') in ('low', 'medium', 'high')
    valid_labels = {f'S{i}' for i in range(1, len(passages[example['id']]) + 1)}
    assert set(structured['citations']).issubset(valid_labels)
    display(structured)
except (ValueError, AssertionError, TypeError) as error:
    print('JSON response needs review:', error)
    print(json_result['answer'])
(REPORTS / f'structured_example_verified_{EMBEDDING_PROVIDER}.json').write_text(
    json.dumps(json_result, indent=2, ensure_ascii=False), encoding='utf-8')


JSON response needs review: 
{
  "answer": "RBC discusses credit risks related to concentration risk, which arises from large exposures that are highly correlated, potentially affecting obligors' ability to meet obligations due to economic or political changes. They actively manage credit exposures and limits to align with their risk appetite and minimize undue concentration risk. Loan forbearance is also addressed, where they may modify loan terms for borrowers in financial difficulty to minimize losses. The bank has specialized groups and policies for managing high-risk borrowers, offering concessions such as rate reductions, payment deferrals, and term extensions. However, the provided filings do not contain specific information on loan-loss provisions.",
  "citations": ["S1", "S4"],
  "sources": [
    {
      "label": "S1",
      "bank": "Royal Bank of Canada",
      "document": "annual_report",
      "section": "Management’s Discussion and Analysis",
      "year": "2024"
    },
  

16042

## 9. Query rewriting

Try the original question and a more search-friendly version. Compare the retrieved passages before assuming rewriting helped.

In [12]:
def rewrite_question(question):
    response = llm.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {'role': 'system', 'content': 'Rewrite the question as a short search query for Canadian bank filings. Preserve bank, year and financial terms. Return only the rewritten query.'},
            {'role': 'user', 'content': question},
        ],
    )
    return (response.choices[0].message.content or question).strip()

query = "What are RBC's main mortgage-related credit risks?"
rewritten = rewrite_question(query)
filters = {'bank_code': 'RBC'}
print('Original:', query)
print('Rewritten:', rewritten)
for name, question in [('original', query), ('rewritten', rewritten)]:
    print('\n', name)
    for hit in retrieve(question, filters, k=5):
        print(source_name(hit), '|', hit['chunk_id'])

Original: What are RBC's main mortgage-related credit risks?
Rewritten: RBC mortgage-related credit risks 2023

 original
Royal Bank of Canada | annual_report | 2024 | Management’s Discussion and Analysis | pages 83-84 | RBC_2024_annual_report_FY_00140
Royal Bank of Canada | annual_report | 2025 | Management’s Discussion and Analysis | pages 82-83 | RBC_2025_annual_report_FY_00133
Royal Bank of Canada | quarterly | 2024 Q2 | Credit risk | pages 26-26 | RBC_2024_quarterly_Q2_00052
Royal Bank of Canada | annual_report | 2024 | Market risk | pages 140-140 | RBC_2024_annual_report_FY_00251
Royal Bank of Canada | annual_report | 2025 | Market risk | pages 138-138 | RBC_2025_annual_report_FY_00243

 rewritten
Royal Bank of Canada | annual_report | 2024 | Management’s Discussion and Analysis | pages 83-84 | RBC_2024_annual_report_FY_00140
Royal Bank of Canada | annual_report | 2025 | Management’s Discussion and Analysis | pages 82-83 | RBC_2025_annual_report_FY_00133
Royal Bank of Canada | qu

## 10. Edge-case tests

These questions check how the system handles unsupported facts, calculations and a question in French. Inspect the results manually; do not mark an unanswered question as a retrieval failure without checking the actual filing.

In [13]:
edge_cases = [
    ('unknown bank', 'What were Bank of Atlantis credit losses in 2025?', {}),
    ('future', 'What will RBC report for credit losses in 2030?', {'bank_code': 'RBC'}),
    ('calculation', 'What percentage did RBC provision for credit losses change year over year?', {'bank_code': 'RBC'}),
    ('ambiguous', 'Did the bank improve its risk profile?', {}),
    ('French', 'Quels risques de crédit la RBC décrit-elle?', {'bank_code': 'RBC'}),
]
EDGE_PATH = REPORTS / f'edge_cases_{EMBEDDING_PROVIDER}.json'
if EDGE_PATH.exists():
    edge_results = json.loads(EDGE_PATH.read_text(encoding='utf-8'))
else:
    edge_results = []
    for category, question, filters in tqdm(edge_cases):
        result = ask_model(question, retrieve(question, filters), 'step_by_step')
        edge_results.append({'category': category, 'question': question, **result})
        EDGE_PATH.write_text(json.dumps(edge_results, indent=2, ensure_ascii=False), encoding='utf-8')
for row in edge_results:
    print('\n', row['category'], '-', row['question'])
    print(row['answer'])


 unknown bank - What were Bank of Atlantis credit losses in 2025?
The provided filings do not contain enough information to answer this question.

 future - What will RBC report for credit losses in 2030?
The provided filings do not contain enough information to answer this question.

 calculation - What percentage did RBC provision for credit losses change year over year?
The provided filings do not contain enough information to answer this question.

 ambiguous - Did the bank improve its risk profile?
The provided filings do not contain enough information to answer this question.

 French - Quels risques de crédit la RBC décrit-elle?
La RBC décrit plusieurs risques de crédit, notamment :

1. Le risque de perte associé à l'incapacité ou au refus d'un débiteur de respecter ses obligations contractuelles en temps voulu, ce qui peut provenir directement d'un débiteur principal ou indirectement d'un débiteur secondaire, ainsi que par le biais d'expositions hors bilan et de risques de cré

Historical results and source provenance are retained in the evaluation reports. Human review ratings remain incomplete. Comparisons requiring 2023 are outside the 2024-2025 corpus.